In [11]:
import numpy as np
import rebound, reboundx

# ---------- Units ----------
sim = rebound.Simulation()
sim.units = ('AU','yr','Msun')
sim.integrator = "whfast"
sim.G = 4*np.pi**2
rebx = reboundx.Extras(sim)

# ---------- Binary ICs ----------
AU_per_Rsun = 1.0/215.032
m1 = m2 = 1.0
a0, e0 = 0.05, 0.05
R1 = R2 = 1.0*AU_per_Rsun

sim.add(m=m1, r=R1, x=-a0*m2/(m1+m2),
        vy=+np.sqrt(sim.G*(m1+m2)/a0)*np.sqrt((1-e0)/(1+e0))*m2/(m1+m2))
sim.add(m=m2, r=R2, x=+a0*m1/(m1+m2),
        vy=-np.sqrt(sim.G*(m1+m2)/a0)*np.sqrt((1-e0)/(1+e0))*m1/(m1+m2))
sim.move_to_com()
sim.dt=sim.particles[1].P/3.
# ---------- Spins & structure ----------
k2_moi = 0.7
I1 = k2_moi*m1*R1**2
I2 = k2_moi*m2*R2**2
P_day = 1.0/365.25
omega0 = 2*np.pi/P_day

for i,(I,R) in enumerate([(I1,R1),(I2,R2)]):
    p = sim.particles[i]
    p.params["Omega"] = np.array([0.0,0.0,omega0])
    p.params["I"]     = I
    p.params["k2"]    = 0.2           # Love number
    p.params["tau"]   = 0.05/365.25    # only used if tides_spin is available

    # magnetic_braking inputs
    p.params["mb_on"]         = 1
    p.params["mb_convective"] = 1
    p.params["mb_tau_conv"]   = 12.5/365.25

# ---------- Add magnetic_braking ----------
mb = rebx.load_operator("magnetic_braking")
rebx.add_operator(mb)
mb.params["mb_Msun"] = 1.0
mb.params["mb_Rsun"] = AU_per_Rsun
mb.params["mb_year"] = 1.0
mb.params["mb_Rossby_sat"] = 0.1
# mb.params["mb_K"] = 1e49  # (optional) stronger for fast demos

# ---------- Try tides_spin; fallback to tides_constant_time_lag ----------
try:
    tides = rebx.load_force('tides_constant_time_lag')
    rebx.add_force(tides)
    use_spin = True
    # spins will evolve self-consistently; nothing else to do
except Exception:
    tides = rebx.load_operator("tides_constant_time_lag")
    rebx.add_operator(tides)
    use_spin = False
    # set tctl params (older operator; no spin evolution)
    for i,p in enumerate(sim.particles[:2]):
        p.params["tctl_k2"]  = 2
        p.params["tctl_tau"] = 0.05/365.25
        # OmegaMag is read by tctl; we will sync it from our Omega vector each step
        Om = p.params["Omega"]; p.params["OmegaMag"] = np.linalg.norm(Om)

def elems():
    p1=sim.particles[1]
    p0=sim.particles[0]
    o = p1.orbit(primary=p0)
    return o.a, o.e

def sync_spin_to_tctl():
    if use_spin: return
    for p in sim.particles[:2]:
        Om = p.params["Omega"]
        p.params["OmegaMag"] = float(np.linalg.norm(Om))

def prot_days(i):
    w = np.linalg.norm(sim.particles[i].params["Omega"])
    return (2*np.pi/w)*365.25

# ---------- Integrate ----------
t_end = 2e5
N = 10
times = np.linspace(0.0, t_end, N)
a_hist, e_hist = [], []
P1_hist, P2_hist = [], []

for t in times:
    sim.integrate(t)
    sync_spin_to_tctl()  # keep tctl aware of magnetic braking spin-down
    a,e = elems()
    a_hist.append(a); e_hist.append(e)
    P1_hist.append(prot_days(0)); P2_hist.append(prot_days(1))

print(("Using tides_spin" if use_spin else "Using tides_constant_time_lag (with Omega→OmegaMag sync)"))
print(f"a: {a_hist[0]:.6f} AU  ->  {a_hist[-1]:.6f} AU")
print(f"e: {e_hist[0]:.4f}     ->  {e_hist[-1]:.4f}")
print(f"P1: {P1_hist[0]:.2f} d ->  {P1_hist[-1]:.2f} d | P2: {P2_hist[0]:.2f} d -> {P2_hist[-1]:.2f} d")


Using tides_spin
a: 0.045652 AU  ->  0.045652 AU
e: 0.0952     ->  0.0952
P1: 1.00 d ->  1.00 d | P2: 1.00 d -> 1.00 d


In [8]:
"""
Simple, framework-free tests for magnetic_braking (REBOUNDx),
progressing from one-star spin-down to a binary with CTL tides
coupled to magnetic braking.

Run:  python test_magnetic_braking.py

Requires: rebound, reboundx, numpy
"""

import math
import numpy as np
import rebound
import reboundx

# ----------------------- Units & constants -----------------------
# We integrate in code units: (length, mass, time) = (AU, Msun, yr)
# with G = (2π)^2 so that a=1 AU around 1 Msun has P=1 yr.
TWOPI = 2.0*math.pi
AU_IN_CM = 1.495978707e13
RSUN_IN_CM = 6.957e10
RSUN_IN_AU = RSUN_IN_CM / AU_IN_CM   # ~0.00465047 AU

def new_sim():
    sim = rebound.Simulation()
    sim.G = TWOPI**2
    sim.integrator = "ias15"
    return sim, reboundx.Extras(sim)

# ----------------------- Helpers -----------------------

def add_magnetic_braking(rebx, K_cgs=2.7e47, Rsun_code=RSUN_IN_AU, year_code=1.0):
    """
    Adds the magnetic_braking operator and sets operator-level scalings
    exactly as described in the attached MB docs:
      - mb_K (cgs)
      - mb_Msun, mb_Rsun, mb_year define Msun, Rsun, year in *code* units
      - mb_Rossby_sat is left at the documented default (0.1), but set explicitly.
    """
    mb = rebx.load_operator("magnetic_braking")
    rebx.add_operator(mb)
    mb.params["mb_K"] = float(K_cgs)
    mb.params["mb_Msun"] = 1.0            # Msun is 1 code mass
    mb.params["mb_Rsun"] = float(Rsun_code)   # Rsun in code lengths
    mb.params["mb_year"] = float(year_code)   # year in code times
    mb.params["mb_Rossby_sat"] = 0.1
    return mb

def set_star_spin_and_MB(p, M=1.0, R=1.0*RSUN_IN_AU, Pspin_days=10.0,
                         k2_gyration=0.1, convective=True,
                         omega_sat=None, tau_conv_days=None, mb_on=True):
    """
    Configure particle 'p' as a (convective) star with:
      - mass M, radius R (code units)
      - moment of inertia I = k2 * M * R^2
      - spin vector Omega along +z from Pspin_days
      - MB per-particle flags (mb_on, mb_convective) and optional saturation data
    """
    p.m = float(M)
    p.r = float(R)
    I = k2_gyration * p.m * p.r**2
    Pspin_yr = Pspin_days / 365.25
    Omega_mag = TWOPI / Pspin_yr
    # Vector parameter 'Omega' is a reb_vec3d; Python can set with array/list
    p.params["Omega"] = np.array([0.0, 0.0, Omega_mag], dtype=float)
    p.params["I"] = float(I)
    p.params["mb_on"] = 1 if mb_on else 0
    p.params["mb_convective"] = 1 if convective else 0
    if omega_sat is not None:
        p.params["mb_omega_sat"] = float(omega_sat)
    if tau_conv_days is not None:
        p.params["mb_tau_conv"] = float(tau_conv_days) / 365.25

def omega_vec_from_params(p):
    """
    Robustly get the 3-vector for 'Omega' from particle params.
    Works whether the binding returns a numpy array, an object with x/y/z,
    or (rarely) a scalar (interpreted as z-directed).
    """
    v = p.params.get("Omega", None)
    if v is None:
        return np.array([0.0, 0.0, 0.0], dtype=float)
    # numpy-like?
    try:
        arr = np.array(v, dtype=float).reshape(-1)
        if arr.size == 3:
            return arr.astype(float)
    except Exception:
        pass
    # object with x,y,z?
    if hasattr(v, "x") and hasattr(v, "y") and hasattr(v, "z"):
        return np.array([float(v.x), float(v.y), float(v.z)], dtype=float)
    # scalar fallback -> along z
    if isinstance(v, (int, float)):
        return np.array([0.0, 0.0, float(v)], dtype=float)
    raise TypeError("params['Omega'] is not a 3-vector in a supported format.")

def omega_mag_of(p):
    return float(np.linalg.norm(omega_vec_from_params(p)))

def sync_OmegaMag_to_Omega(sim, star_indices):
    """Copy |Omega| → particle.params['OmegaMag'] for CTL tides."""
    for i in star_indices:
        p = sim.particles[i]
        p.params["OmegaMag"] = omega_mag_of(p)

def integrate_with_spin_sync(sim, tmax, nsteps, star_indices):
    """Step the sim nsteps to tmax, syncing OmegaMag before each step."""
    dt = (tmax - sim.t)/float(nsteps)
    for _ in range(nsteps):
        sync_OmegaMag_to_Omega(sim, star_indices)
        sim.integrate(sim.t + dt)

def semimajor_axis_about(sim, i, j):
    """a of particle i about primary j using documented Python API."""
    return sim.particles[i].a

def assert_monotone_decrease(seq, msg):
    if any(seq[k+1] > seq[k] for k in range(len(seq)-1)):
        raise AssertionError(msg + f" | first few={seq[:6]}")

def approx_equal(a, b, rtol=0.05, msg=""):
    if not (abs(a-b) <= rtol*max(1.0, abs(b))):
        raise AssertionError(f"{msg} | got {a:.6g}, expected {b:.6g} (rtol={rtol})")

# ----------------------- Tests -----------------------

def test_01_one_star_unsaturated_closed_form_two_step():
    """
    One isolated star, unsaturated MB. We do *two identical* intervals Δt:
      1) measure w0 -> w1 to infer C from your closed form,
      2) predict w2 from w1 using the same formula, and compare to the actual w2.
    This self-calibrates away unit-conversion guesses.
    """
    sim, rebx = new_sim()
    sim.add(m=1.0)               # single star
    add_magnetic_braking(rebx, K_cgs=1.0e53)   # make effect visible
    set_star_spin_and_MB(sim.particles[0], Pspin_days=10.0, omega_sat=float("inf"))

    dt = 5.0e4  # years
    w0 = omega_mag_of(sim.particles[0])

    sim.integrate(dt)    # step 1
    w1 = omega_mag_of(sim.particles[0])

    # infer C from the unsaturated formula: w1 = w0 / sqrt(1 + 2 C w0^2 dt)
    denom = (w0/w1)**2
    C_inferred = (denom - 1.0)/(2.0*w0*w0*dt)

    # predicted second step
    w2_pred = w1 / math.sqrt(1.0 + 2.0*C_inferred*w1*w1*dt)

    sim.integrate(2*dt)  # step 2 (total time 2*dt)
    w2 = omega_mag_of(sim.particles[0])

    assert_monotone_decrease([w0, w1, w2], "Unsaturated spin should decrease monotonically")
    approx_equal(w2, w2_pred, rtol=0.03,
                 msg="Unsaturated closed-form two-step prediction (~3%)")

def test_02_one_star_saturated_closed_form_two_step():
    """
    One star in saturated regime (Omega > omega_sat).
    Use two identical steps Δt, infer (C*omega_sat^2) from step 1, predict step 2.
    Exponential decay: w1 = w0 * exp(-C ω_sat^2 Δt)
    """
    sim, rebx = new_sim()
    sim.add(m=1.0)
    add_magnetic_braking(rebx, K_cgs=1.0e53)
    set_star_spin_and_MB(sim.particles[0], Pspin_days=1.0)  # rapid rotator
    sim.particles[0].params["mb_omega_sat"] = 50.0          # set saturation threshold (rad/yr)

    dt = 2.0e4
    w0 = omega_mag_of(sim.particles[0])

    sim.integrate(dt)
    w1 = omega_mag_of(sim.particles[0])

    # infer lambda = C * omega_sat^2
    lam = -math.log(max(1e-300, w1/w0))/dt
    w2_pred = w1 * math.exp(-lam*dt)

    sim.integrate(2*dt)
    w2 = omega_mag_of(sim.particles[0])

    assert_monotone_decrease([w0, w1, w2], "Saturated spin should decrease monotonically")
    approx_equal(w2, w2_pred, rtol=0.05,
                 msg="Saturated exponential two-step prediction (~5%)")

def test_03_activation_guards_disable_braking():
    """
    The MB docs specify that *both* mb_on=1 and mb_convective=1 are required.
    Disable either and |Omega| must remain unchanged.
    """
    # convective = 0
    sim, rebx = new_sim()
    sim.add(m=1.0)
    add_magnetic_braking(rebx, K_cgs=1.0e55)
    set_star_spin_and_MB(sim.particles[0], Pspin_days=3.0, convective=False, mb_on=True)
    sim.integrate(sim.t+1.0e5)
    w0 = omega_mag_of(sim.particles[0])
    sim.integrate(sim.t+2.0e5)    
    w1 = omega_mag_of(sim.particles[0])
    if abs(w1 - w0) > 1e-12:
        print (w1 , w0)
        raise AssertionError("Braking should be disabled when mb_convective=0")

    # mb_on = 0
    sim, rebx = new_sim()
    sim.add(m=1.0)
    add_magnetic_braking(rebx, K_cgs=1.0e55)
    set_star_spin_and_MB(sim.particles[0], Pspin_days=3.0, convective=True, mb_on=False)
    sim.integrate(sim.t+1.0e5)
    w0 = omega_mag_of(sim.particles[0])
    sim.integrate(sim.t+2.0e5)    
    w1 = omega_mag_of(sim.particles[0])
    if abs(w1 - w0) > 1e-12:
        print (w1 , w0)
        raise AssertionError("Braking should be disabled when mb_on=0")

def test_04_binary_no_tides_orbit_constant_with_MB():
    """
    Equal-mass circular binary, MB on star 0 only. Without tides, orbital elements
    should not drift (MB acts on spins only).
    """
    sim, rebx = new_sim()
    a0 = 0.1
    sim.add(m=1.0, r=RSUN_IN_AU)
    sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)
    sim.move_to_com()

    add_magnetic_braking(rebx, K_cgs=1.0e55)
    set_star_spin_and_MB(sim.particles[0], Pspin_days=3.0)
    sim.particles[1].params["mb_on"] = 0  # no braking on star 1

    a_start = semimajor_axis_about(sim, 1, 0)
    sim.integrate(500.0)
    a_end = semimajor_axis_about(sim, 1, 0)
    if abs(a_end - a_start) > 1e-5 * a_start:
        raise AssertionError("Orbit should remain constant without tides (spin-only torque).")

def test_05_binary_CTL_only_shrinks():
    """
    CTL tides alone with sub-synchronous spin (OmegaMag=0) should shrink a.
    """
    sim, rebx = new_sim()
    a0 = 0.05
    sim.add(m=1.0, r=RSUN_IN_AU)
    sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)
    sim.move_to_com()

    tctl = rebx.load_force("tides_constant_time_lag"); rebx.add_force(tctl)
    for p in (sim.particles[0], sim.particles[1]):
        p.params["tctl_k2"] = 0.03
        p.params["tctl_tau"] = 1e-4      # large to see effect quickly (yrs)
    #    p.params["OmegaMag"] = 0.0       # sub-synchronous

    a_start = semimajor_axis_about(sim, 1, 0)
    sim.integrate(10.0)
    a_end = semimajor_axis_about(sim, 1, 0)
    if not (a_end < a_start):
        raise AssertionError("CTL-only case should shrink a for sub-synchronous spins.")

def test_06_binary_CTL_plus_MB_shrinks_more():
    """
    Same setup as test_05, but add MB to star 0 and *couple* by syncing
    OmegaMag ← |Omega| before each step. Expect *more* orbital decay than CTL-only.
    """
    # --- Control (CTL only) ---
    ctl, xctl = new_sim()
    a0 = 0.05
    ctl.add(m=1.0, r=RSUN_IN_AU)
    ctl.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)
    ctl.move_to_com()
    tctl_c = xctl.load_force("tides_constant_time_lag"); xctl.add_force(tctl_c)
    for p in (ctl.particles[0], ctl.particles[1]):
        p.params["tctl_k2"] = 0.03
        p.params["tctl_tau"] = 1e-4
    #    p.params["OmegaMag"] = 0.0
    a0_ctl = semimajor_axis_about(ctl, 1, 0)
    ctl.integrate(300.0)
    a1_ctl = semimajor_axis_about(ctl, 1, 0)

    # --- CTL + MB (coupled) ---
    sim, rebx = new_sim()
    sim.add(m=1.0, r=RSUN_IN_AU)
    sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)
    sim.move_to_com()
    tctl = rebx.load_force("tides_constant_time_lag"); rebx.add_force(tctl)
    add_magnetic_braking(rebx, K_cgs=1.0e53)  # braking on
    # initialize slow spins and enable MB on primary only
    set_star_spin_and_MB(sim.particles[0], Pspin_days=30.0)
    set_star_spin_and_MB(sim.particles[1], Pspin_days=30.0, mb_on=False)

    for p in (sim.particles[0], sim.particles[1]):
        p.params["tctl_k2"] = 0.03
        p.params["tctl_tau"] = 1e-4

    a0_mb = semimajor_axis_about(sim, 1, 0)
    integrate_with_spin_sync(sim, tmax=300.0, nsteps=200, star_indices=(0,1))
    a1_mb = semimajor_axis_about(sim, 1, 0)

    if not (a1_mb < a1_ctl):
        print (a1_mb,a1_ctl)
        raise AssertionError("CTL+MB did not shrink the orbit more than CTL-only.")

def test_07_mbK_scaling_monotone():
    """
    Stronger mb_K -> stronger spin-down over a fixed interval.
    """
    def final_omega(K):
        sim, rebx = new_sim()
        sim.add(m=1.0)
        add_magnetic_braking(rebx, K_cgs=K)
        set_star_spin_and_MB(sim.particles[0], Pspin_days=5.0)
        sim.integrate(2.0e2)
        return omega_mag_of(sim.particles[0])

    w_loK = final_omega(1.0e51)
    w_hiK = final_omega(1.0e53)
    if not (w_hiK < w_loK):
        raise AssertionError("Larger mb_K should yield smaller final Omega.")

def test_08_cross_saturation_then_slowdown():
    """
    Start above omega_sat, integrate long enough to cross below it.
    Heuristic check: after crossing, decay is slower than *pure exponential*
    with the original (saturated) decay constant.
    """
    sim, rebx = new_sim()
    sim.add(m=1.0)
    add_magnetic_braking(rebx, K_cgs=1.0e53)
    set_star_spin_and_MB(sim.particles[0], Pspin_days=1.0)
    sim.particles[0].params["mb_omega_sat"] = 30.0  # rad/yr

    t1 = 2.0e5
    w0 = omega_mag_of(sim.particles[0])
    sim.integrate(t1)
    w1 = omega_mag_of(sim.particles[0])

    # infer saturated decay constant from first interval
    lam = -math.log(max(1e-300, w1/w0))/t1

    # advance much further; once unsaturated, decay slows relative to this exponential
    t2 = 4.0e5
    sim.integrate(t2)
    w2 = omega_mag_of(sim.particles[0])

    w2_exp_from_t1 = w1 * math.exp(-lam*(t2 - t1))
    if not (w2 > w2_exp_from_t1):
        print (w2 , w2_exp_from_t1)
        raise AssertionError("After crossing saturation, decay should be slower than exp. extrapolation.")

# ----------------------- Runner -----------------------

def main():
    tests = [
        ("01_one_star_unsaturated_closed_form_two_step", test_01_one_star_unsaturated_closed_form_two_step),
        ("02_one_star_saturated_closed_form_two_step",   test_02_one_star_saturated_closed_form_two_step),
        ("03_activation_guards_disable_braking",         test_03_activation_guards_disable_braking),
        ("04_binary_no_tides_orbit_constant_with_MB",    test_04_binary_no_tides_orbit_constant_with_MB),
        ("05_binary_CTL_only_shrinks",                   test_05_binary_CTL_only_shrinks),
        ("06_binary_CTL_plus_MB_shrinks_more",           test_06_binary_CTL_plus_MB_shrinks_more),
        ("07_mbK_scaling_monotone",                      test_07_mbK_scaling_monotone),
        ("08_cross_saturation_then_slowdown",            test_08_cross_saturation_then_slowdown),
    ]
    passed = 0
    for name, fn in tests:
        try:
            fn()
            print(f"[PASS] {name}")
            passed += 1
        except AssertionError as e:
            print(f"[FAIL] {name}: {e}")
        except Exception as e:
            print(f"[ERROR] {name}: {e}")
    print(f"\n{passed}/{len(tests)} tests passed.")

if __name__ == "__main__":
    main()


[PASS] 01_one_star_unsaturated_closed_form_two_step
[PASS] 02_one_star_saturated_closed_form_two_step
1.124934971247958 1.948440311904755
[FAIL] 03_activation_guards_disable_braking: Braking should be disabled when mb_convective=0
[PASS] 04_binary_no_tides_orbit_constant_with_MB
[PASS] 05_binary_CTL_only_shrinks
0.04742679601507483 0.047101053576074754
[FAIL] 06_binary_CTL_plus_MB_shrinks_more: CTL+MB did not shrink the orbit more than CTL-only.
[PASS] 07_mbK_scaling_monotone
[PASS] 08_cross_saturation_then_slowdown

6/8 tests passed.


In [30]:
tests = [
    ("01_one_star_unsaturated_closed_form_two_step", test_01_one_star_unsaturated_closed_form_two_step),
    ("02_one_star_saturated_closed_form_two_step",   test_02_one_star_saturated_closed_form_two_step),
    ("03_activation_guards_disable_braking",         test_03_activation_guards_disable_braking),
    ("04_binary_no_tides_orbit_constant_with_MB",    test_04_binary_no_tides_orbit_constant_with_MB),
    ("05_binary_CTL_only_shrinks",                   test_05_binary_CTL_only_shrinks),
    ("06_binary_CTL_plus_MB_shrinks_more",           test_06_binary_CTL_plus_MB_shrinks_more),
    ("07_mbK_scaling_monotone",                      test_07_mbK_scaling_monotone),
    ("08_cross_saturation_then_slowdown",            test_08_cross_saturation_then_slowdown),
]

tests[5][1]()

TypeError: 'float' object is not subscriptable

In [1]:
"""
Physical tests for magnetic braking (REBOUNDx) and its coupling to
tides_constant_time_lag (CTL). No external test framework: just functions,
asserts, and prints.

Run:  python physical_tests_magnetic_braking.py

Requires: rebound, reboundx, numpy
"""

import math
import numpy as np
import rebound
import reboundx

# ----------------------- Units & constants -----------------------
# Code units: (AU, Msun, yr), with G=(2π)^2 so P(1 AU, 1 Msun)=1 yr
TWOPI = 2.0*math.pi
G_CODE = TWOPI**2
AU_IN_CM = 1.495978707e13
RSUN_IN_CM = 6.957e10
RSUN_IN_AU = RSUN_IN_CM / AU_IN_CM   # ~0.00465047 AU

# ----------------------- Simulation helpers -----------------------

def new_sim():
    sim = rebound.Simulation()
    sim.G = G_CODE
    sim.integrator = "ias15"
    return sim, reboundx.Extras(sim)

def add_magnetic_braking(rebx, K_cgs=1.0e53, Rsun_code=RSUN_IN_AU, year_code=1.0):
    """
    Load the magnetic_braking operator with explicit unit scalings.
    """
    mb = rebx.load_operator("magnetic_braking")
    rebx.add_operator(mb)
    mb.params["mb_K"] = float(K_cgs)      # cgs constant; operator handles unit conversion
    mb.params["mb_Msun"] = 1.0            # Msun in code units
    mb.params["mb_Rsun"] = float(Rsun_code)
    mb.params["mb_year"] = float(year_code)
    mb.params["mb_Rossby_sat"] = 0.1      # explicit default
    return mb

def set_star_spin_and_MB(p, M=1.0, R=RSUN_IN_AU, Pspin_days=10.0,
                         k2_gyration=0.1, spin_axis=(0,0,1),
                         convective=True, mb_on=True,
                         omega_sat=None, tau_conv_days=None):
    """
    Configure a star:
      - mass M, radius R (code units)
      - moment of inertia I = k2 * M * R^2
      - spin vector Omega = |Omega| * unit(spin_axis), with |Omega| from Pspin_days
      - MB per-particle flags and optional saturation inputs
    """
    p.m = float(M)
    p.r = float(R)
    I = k2_gyration * p.m * p.r**2
    Pspin_yr = Pspin_days / 365.25
    Om_mag = TWOPI / Pspin_yr
    ax = np.array(spin_axis, dtype=float)
    if np.linalg.norm(ax) == 0:
        ax = np.array([0.0,0.0,1.0])
    ax = ax / np.linalg.norm(ax)
    p.params["Omega"] = (ax * Om_mag).astype(float)
    p.params["I"] = float(I)
    p.params["mb_on"] = 1 if mb_on else 0
    p.params["mb_convective"] = 1 if convective else 0
    if omega_sat is not None:
        p.params["mb_omega_sat"] = float(omega_sat)
    if tau_conv_days is not None:
        p.params["mb_tau_conv"] = float(tau_conv_days)/365.25

def omega_vec_from_params(p):
    v = p.params.get("Omega", None)
    if v is None: return np.array([0.0,0.0,0.0], dtype=float)
    # try numpy-like
    try:
        arr = np.array(v, dtype=float).reshape(-1)
        if arr.size == 3: return arr
    except Exception:
        pass
    # object with x,y,z
    if hasattr(v, "x") and hasattr(v, "y") and hasattr(v, "z"):
        return np.array([float(v.x), float(v.y), float(v.z)], dtype=float)
    # scalar fallback -> z-axis
    if isinstance(v, (int, float)):
        return np.array([0.0, 0.0, float(v)], dtype=float)
    raise TypeError("Omega param is not a 3-vector in a supported format.")

def omega_mag(p): return float(np.linalg.norm(omega_vec_from_params(p)))

def L_spin_vec(p):  return float(p.params["I"]) * omega_vec_from_params(p)
def L_spin_mag(p):  return float(np.linalg.norm(L_spin_vec(p)))
def E_rot(p):       return 0.5*float(p.params["I"])*(omega_mag(p)**2)

def sync_OmegaMag_to_Omega(sim, star_indices):
    """Couple MB → CTL by copying |Omega| to OmegaMag for each star."""
    for i in star_indices:
        sim.particles[i].params["OmegaMag"] = omega_mag(sim.particles[i])

def integrate_with_spin_sync(sim, tmax, nsteps, star_indices):
    dt = (tmax - sim.t)/float(nsteps)
    for _ in range(nsteps):
        sync_OmegaMag_to_Omega(sim, star_indices)
        sim.integrate(sim.t + dt)

# ----------------------- Two-body diagnostics -----------------------

def rel_state(sim, i=0, j=1):
    """Relative position and velocity of j around i."""
    pi, pj = sim.particles[i], sim.particles[j]
    r = np.array([pj.x - pi.x, pj.y - pi.y, pj.z - pi.z], dtype=float)
    v = np.array([pj.vx - pi.vx, pj.vy - pi.vy, pj.vz - pi.vz], dtype=float)
    return r, v

def J_orbital_vec(sim, i=0, j=1):
    """Total orbital angular momentum vector of the two-body system."""
    pi, pj = sim.particles[i], sim.particles[j]
    mu = pi.m*pj.m/(pi.m+pj.m)
    r, v = rel_state(sim, i, j)
    return mu * np.cross(r, v)

def J_orbital_mag(sim, i=0, j=1): return float(np.linalg.norm(J_orbital_vec(sim, i, j)))

def J_total_vec(sim, star_indices=(0,1)):
    J = J_orbital_vec(sim, 0, 1)
    for k in star_indices:
        J = J + L_spin_vec(sim.particles[k])
    return J

def a_e(sim, i=1, j=0):
    """a,e of particle i around j using documented API."""
    orb = sim.particles[i]#.calculate_orbit(primary=sim.particles[j])
    return float(orb.a), float(orb.e)

def E_orbital(sim, i=0, j=1):
    """Total orbital energy (kinetic + potential of two bodies)."""
    pi, pj = sim.particles[i], sim.particles[j]
    r, v = rel_state(sim, i, j)
    rmag = float(np.linalg.norm(r))
    # Kinetic energy in COM frame: μ v_rel^2/2 ; but easier via two-body: 
    mu = pi.m*pj.m/(pi.m+pj.m)
    K = 0.5*mu*float(np.dot(v, v))
    U = -sim.G * pi.m * pj.m / rmag
    return K + U

def n_mean_motion(sim, a, m_total):
    """Mean motion n (rad/yr) for semimajor axis a and total mass."""
    return math.sqrt(sim.G * m_total / (a**3))

def roche_lobe_radius(a, q):
    """Eggleton Roche lobe radius for star of mass M1 with companion M2: q=M1/M2."""
    q13 = q**(1.0/3.0)
    return a * 0.49*q13*q13 / (0.6*q13*q13 + math.log(1.0 + q13))

def assert_outside_roche(sim, margin=3.0, i=0, j=1):
    """Assert both stars well inside their Roche lobes: R < RL/margin."""
    a, _ = a_e(sim, i=j, j=i)  # a of j around i equals a of i around j
    M1, M2 = sim.particles[i].m, sim.particles[j].m
    RL1 = roche_lobe_radius(a, q=M1/M2)
    RL2 = roche_lobe_radius(a, q=M2/M1)
    if not (sim.particles[i].r < RL1/margin and sim.particles[j].r < RL2/margin):
        raise AssertionError(f"Roche-lobe violation: R1={sim.particles[i].r:.3e}, RL1={RL1:.3e}; "
                             f"R2={sim.particles[j].r:.3e}, RL2={RL2:.3e} (margin={margin})")

def monotone_decrease(vals): return all(vals[k+1] <= vals[k] for k in range(len(vals)-1))
def monotone_increase(vals): return all(vals[k+1] >= vals[k] for k in range(len(vals)-1))

# ----------------------- PHYSICAL TESTS -----------------------

def phys_01_single_star_MB_removes_L_and_E_preserves_axis():
    """
    Magnetic braking removes spin angular momentum and rotational energy,
    and (for an isotropic wind torque) does not tilt the spin axis:
      - |L_spin| decreases monotonically
      - E_rot decreases monotonically
      - unit(Omega) direction stays fixed (no precession from MB)
    """
    sim, rebx = new_sim()
    sim.add(m=1.0)
    add_magnetic_braking(rebx, K_cgs=1.0e53)
    # Tilted spin axis: 30° from +z in x–z plane
    theta = math.radians(30.0)
    set_star_spin_and_MB(sim.particles[0], Pspin_days=5.0,
                         spin_axis=(math.sin(theta), 0.0, math.cos(theta)))

    times = np.linspace(0.0, 1.0e5, 20)
    Ls, Er, dirs = [], [], []
    for t in times:
        sim.integrate(t)
        Om = omega_vec_from_params(sim.particles[0])
        dirs.append(Om/np.linalg.norm(Om))
        Ls.append(L_spin_mag(sim.particles[0]))
        Er.append(E_rot(sim.particles[0]))

    if not monotone_decrease(Ls[:5] + Ls[-5:]):  # relaxed monotonicity (solver noise)
        raise AssertionError("MB should monotonically reduce |L_spin|.")
    if not monotone_decrease(Er[:5] + Er[-5:]):
        raise AssertionError("MB should monotonically reduce rotational energy.")

    # Axis constancy: first and last directions should be nearly identical
    c = float(np.dot(dirs[0], dirs[-1]))
    if c < 1.0 - 1e-6:  # tiny numerical drift tolerated
        raise AssertionError(f"Spin axis should be preserved by MB (dot={c:.6g}).")

def phys_02_single_star_no_MB_conserves_L_and_E():
    """
    With MB disabled physically (mb_on=0), in absence of other spin torques,
    |L_spin| and E_rot should remain constant.
    """
    sim, rebx = new_sim()
    sim.add(m=1.0)
    add_magnetic_braking(rebx, K_cgs=1.0e55)  # irrelevant if mb_on=0
    set_star_spin_and_MB(sim.particles[0], Pspin_days=5.0, mb_on=False)

    w0 = omega_mag(sim.particles[0]); L0 = L_spin_mag(sim.particles[0]); E0 = E_rot(sim.particles[0])
    sim.integrate(1.0e5)
    w1 = omega_mag(sim.particles[0]); L1 = L_spin_mag(sim.particles[0]); E1 = E_rot(sim.particles[0])

    if abs(L1-L0) > 1e-12 or abs(E1-E0) > 1e-12 or abs(w1-w0) > 1e-12:
        raise AssertionError("Without MB, spin angular momentum and energy must be constant.")

def phys_03_single_star_I_scaling_slows_spindown():
    """
    Larger moment of inertia -> slower change in Omega over same interval:
    dOmega/dt ~ tau/I (tau from MB). Compare k2=0.1 vs 0.2.
    """
    def final_omega(k2):
        sim, rebx = new_sim()
        sim.add(m=1.0)
        add_magnetic_braking(rebx, K_cgs=1.0e53)
        set_star_spin_and_MB(sim.particles[0], Pspin_days=3.0, k2_gyration=k2)
        sim.integrate(5.0e4)
        return omega_mag(sim.particles[0])
    w_smallI = final_omega(0.1)
    w_largeI = final_omega(0.2)
    if not (w_largeI > w_smallI):
        raise AssertionError("Larger I should retain a larger Omega after equal time.")

def phys_04_single_star_saturated_braking_is_weaker_than_unsaturated():
    """
    For the same initial fast spin, saturated MB (omega_sat finite) reduces Omega
    *less* than unsaturated (omega_sat = inf) over the same time.
    """
    def final_omega(omega_sat):
        sim, rebx = new_sim()
        sim.add(m=1.0)
        add_magnetic_braking(rebx, K_cgs=1.0e53)
        set_star_spin_and_MB(sim.particles[0], Pspin_days=1.0)  # very fast
        if omega_sat is not None:
            sim.particles[0].params["mb_omega_sat"] = float(omega_sat)
        sim.integrate(5.0e4)
        return omega_mag(sim.particles[0])

    w_unsat = final_omega(None)       # unsaturated
    w_sat   = final_omega(50.0)       # saturated at moderate threshold
    if not (w_sat > w_unsat):
        raise AssertionError("Saturated MB should produce *less* spin-down than unsaturated.")

def phys_05_binary_MB_only_keeps_orbit_constant():
    """
    Physical expectation: pure MB torques spin, not orbit.
    In a detached binary with MB on star 0 only and no tides:
      - a and J_orb should remain constant (up to tiny numeric drift).
      - total J should drop exactly by ΔL_spin.
    """
    sim, rebx = new_sim()
    a0 = 0.1
    sim.add(m=1.0, r=RSUN_IN_AU)
    sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)
    sim.move_to_com()

    add_magnetic_braking(rebx, K_cgs=1.0e55)   # strong to make a visible spin change
    set_star_spin_and_MB(sim.particles[0], Pspin_days=3.0)   # MB on
    set_star_spin_and_MB(sim.particles[1], Pspin_days=3.0, mb_on=False)  # MB off

    a_start, e_start = a_e(sim, 1, 0)
    Jorb_start = J_orbital_mag(sim)
    Jtot_start = float(np.linalg.norm(J_total_vec(sim, (0,1))))
    Lspin_start = L_spin_mag(sim.particles[0]) + L_spin_mag(sim.particles[1])

    sim.integrate(2.0e3)  # years (short; MB shouldn't touch orbit)
    a_end, e_end = a_e(sim, 1, 0)
    Jorb_end = J_orbital_mag(sim)
    Jtot_end = float(np.linalg.norm(J_total_vec(sim, (0,1))))
    Lspin_end = L_spin_mag(sim.particles[0]) + L_spin_mag(sim.particles[1])

    assert_outside_roche(sim, margin=5.0)

    if abs(a_end - a_start) > 1e-6*a_start or abs(Jorb_end - Jorb_start) > 1e-6*Jorb_start:
        raise AssertionError("With MB only, orbit (a, J_orb) must remain constant.")
    # Budget: ΔJ_total ≈ ΔL_spin  (since orbit unchanged)
    if abs((Jtot_end - Jtot_start) - (Lspin_end - Lspin_start)) > 1e-9*max(1.0, Jtot_start):
        raise AssertionError("With MB only, ΔJ_total should equal ΔL_spin.")

def phys_06_binary_CTL_only_subsync_inspiral():
    """
    CTL-only: For sub-synchronous spin (OmegaMag < n), the orbit loses energy and a decreases.
    Only star 0 raises tides to avoid mixed effects.
    """
    sim, rebx = new_sim()
    a0 = 0.05
    sim.add(m=1.0, r=RSUN_IN_AU)            # star 0
    sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)  # star 1
    sim.move_to_com()

    tctl = rebx.load_force("tides_constant_time_lag"); rebx.add_force(tctl)
    # Only star 0 dissipates
    sim.particles[0].params["tctl_k2"] = 0.03
    sim.particles[0].params["tctl_tau"] = 1e-4
    sim.particles[1].params["tctl_k2"] = 0.0
    sim.particles[1].params["tctl_tau"] = 0.0

    # Sub-synchronous: OmegaMag = 0
#    sim.particles[0].params["OmegaMag"] = 0.0

    a_start, _ = a_e(sim, 1, 0)
    sim.integrate(2.0e2)
    a_end, _ = a_e(sim, 1, 0)

    assert_outside_roche(sim, margin=5.0)
    if not (a_end < a_start):
        raise AssertionError("CTL-only + sub-synchronous spin should shrink the orbit (a decreases).")

def phys_07_binary_CTL_only_supersync_expansion():
    """
    CTL-only: For super-synchronous spin (OmegaMag > n), tidal torques transfer spin to orbit
    and a should increase (expansion). Only star 0 dissipates.
    """
    sim, rebx = new_sim()
    a0 = 0.05
    sim.add(m=1.0, r=RSUN_IN_AU)
    sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)
    sim.move_to_com()

    tctl = rebx.load_force("tides_constant_time_lag"); rebx.add_force(tctl)
    sim.particles[0].params["tctl_k2"] = 0.03
    sim.particles[0].params["tctl_tau"] = 1e-4
    sim.particles[1].params["tctl_k2"] = 0.0
    sim.particles[1].params["tctl_tau"] = 0.0

    # Super-synchronous: set OmegaMag = 2 n
    a_start, _ = a_e(sim, 1, 0)
    n0 = n_mean_motion(sim, a_start, sim.particles[0].m + sim.particles[1].m)
    sim.particles[0].params["OmegaMag"] = 2.0*n0

    sim.integrate(2.0e3)
    a_end, _ = a_e(sim, 1, 0)

    assert_outside_roche(sim, margin=5.0)
    if not (a_end > a_start):
        raise AssertionError("CTL-only + super-synchronous spin should expand the orbit (a increases).")

def phys_08_binary_CTL_plus_MB_accelerates_inspiral():
    """
    CTL + MB: With both stars initially sub-synchronous, MB keeps spins low,
    sustaining strong tidal torques -> faster inspiral than CTL-only.
    """
    # --- Control: CTL only ---
    ctl, xctl = new_sim()
    a0 = 0.05
    ctl.add(m=1.0, r=RSUN_IN_AU)
    ctl.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)
    ctl.move_to_com()
    fctl = xctl.load_force("tides_constant_time_lag"); xctl.add_force(fctl)
    for p in (ctl.particles[0], ctl.particles[1]):
        p.params["tctl_k2"] = 0.03
        p.params["tctl_tau"] = 1e-4
#        p.params["OmegaMag"] = 0.0  # sub-synchronous
    a0_ctl, _ = a_e(ctl, 1, 0)
    ctl.integrate(2.0e2)
    a1_ctl, _ = a_e(ctl, 1, 0)

    # --- CTL + MB (MB on both) ---
    sim, rebx = new_sim()
    sim.add(m=1.0, r=RSUN_IN_AU)
    sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)
    sim.move_to_com()
    tctl = rebx.load_force("tides_constant_time_lag"); rebx.add_force(tctl)
    add_magnetic_braking(rebx, K_cgs=1.0e53)
    # Initialize slow spins; MB on both
    set_star_spin_and_MB(sim.particles[0], Pspin_days=30.0)
    set_star_spin_and_MB(sim.particles[1], Pspin_days=30.0)

    for p in (sim.particles[0], sim.particles[1]):
        p.params["tctl_k2"] = 0.03
        p.params["tctl_tau"] = 1e-4

    a0_mb, _ = a_e(sim, 1, 0)
    integrate_with_spin_sync(sim, tmax=2.0e2, nsteps=200, star_indices=(0,1))
    a1_mb, _ = a_e(sim, 1, 0)

    assert_outside_roche(sim, margin=5.0)
    if not (a1_mb < a1_ctl):
        raise AssertionError("CTL+MB should shrink a *more* than CTL-only in sub-synchronous regime.")

def phys_09_binary_CTL_plus_MB_flip_supersync_to_subsync():
    """
    Start super-synchronous with CTL+MB on the primary only.
    Expect: initial expansion (a increases), then as MB slows the spin below n,
    the orbit should turn around and begin to shrink.
    """
    sim, rebx = new_sim()
    a0 = 0.05
    sim.add(m=1.0, r=RSUN_IN_AU)                     # primary (braked)
    sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)        # secondary
    sim.move_to_com()
    tctl = rebx.load_force("tides_constant_time_lag"); rebx.add_force(tctl)
    add_magnetic_braking(rebx, K_cgs=1.0e55)         # strong MB to force a flip within run

    # CTL on star 0 only
    sim.particles[0].params["tctl_k2"] = 0.03
    sim.particles[0].params["tctl_tau"] = 1e-4
    sim.particles[1].params["tctl_k2"] = 0.0
    sim.particles[1].params["tctl_tau"] = 0.0

    # primary initial super-synchronous spin: Omega ~ 3 n
    a_init, _ = a_e(sim, 1, 0)
    n0 = n_mean_motion(sim, a_init, sim.particles[0].m + sim.particles[1].m)
    set_star_spin_and_MB(sim.particles[0], Pspin_days=1.0)   # very fast
    # Overwrite with exact multiple of n:
    Om_vec = omega_vec_from_params(sim.particles[0])
    sim.particles[0].params["Omega"] = (Om_vec/np.linalg.norm(Om_vec))* (3.0*n0)
    # Secondary: no MB to isolate effect
    set_star_spin_and_MB(sim.particles[1], Pspin_days=30.0, mb_on=False)

    # Evolve with coupling; track a(t)
    times = np.linspace(0.0, 5.0e2, 120)
    a_series = []
    for t in times:
        integrate_with_spin_sync(sim, t, nsteps=1, star_indices=(0,1))
        a_now, _ = a_e(sim, 1, 0); a_series.append(a_now)

    assert_outside_roche(sim, margin=5.0)

    # Find initial trend and final trend
    if len(a_series) < 10: raise AssertionError("Insufficient sampling.")
    early = a_series[:10]; late = a_series[-10:]
    if not monotone_increase(early):
        raise AssertionError("Early-time behavior should be expansion (a increasing).")
    if not monotone_decrease(late):
        raise AssertionError("Late-time behavior should be decay (a decreasing) after spin slows below n.")

def phys_10_binary_CTL_only_eccentric_circularization():
    """
    Detached eccentric binary with sub-synchronous spins (CTL only).
    Expect e to damp (circularization), with a typically decreasing.
    """
    sim, rebx = new_sim()
    a0 = 0.08
    sim.add(m=1.0, r=RSUN_IN_AU)
    sim.add(m=1.0, a=a0, e=0.3, r=RSUN_IN_AU)
    sim.move_to_com()

    tctl = rebx.load_force("tides_constant_time_lag"); rebx.add_force(tctl)
    for p in (sim.particles[0], sim.particles[1]):
        p.params["tctl_k2"] = 0.03
        p.params["tctl_tau"] = 1e-4
#        p.params["OmegaMag"] = 0.0

    e_series, a_series = [], []
    times = np.linspace(0.0, 5.0e2, 60)
    for t in times:
        sim.integrate(t)
        a_now, e_now = a_e(sim, 1, 0)
        a_series.append(a_now); e_series.append(e_now)

    assert_outside_roche(sim, margin=5.0)
    if not monotone_decrease(e_series[:5] + e_series[-5:]):
        raise AssertionError("CTL-only should circularize the orbit (e decreases).")

def phys_11_binary_CTL_plus_MB_vs_one_or_both_braked():
    """
    With CTL engaged, braking both stars keeps both spins sub-synchronous, increasing
    the net tidal drain of orbital energy compared to braking only one star.
    Expect: a_both < a_one after the same time.
    """
    def run(brake_primary=True, brake_secondary=False):
        sim, rebx = new_sim()
        a0 = 0.05
        sim.add(m=1.0, r=RSUN_IN_AU)
        sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)
        sim.move_to_com()
        tctl = rebx.load_force("tides_constant_time_lag"); rebx.add_force(tctl)
        add_magnetic_braking(rebx, K_cgs=1.0e53)

        set_star_spin_and_MB(sim.particles[0], Pspin_days=30.0, mb_on=brake_primary)
        set_star_spin_and_MB(sim.particles[1], Pspin_days=30.0, mb_on=brake_secondary)

        for p in (sim.particles[0], sim.particles[1]):
            p.params["tctl_k2"] = 0.03
            p.params["tctl_tau"] = 1e-4

        integrate_with_spin_sync(sim, tmax=2.0e2, nsteps=200, star_indices=(0,1))
        a_end, _ = a_e(sim, 1, 0)
        assert_outside_roche(sim, margin=5.0)
        return a_end

    a_one  = run(brake_primary=True,  brake_secondary=False)
    a_both = run(brake_primary=True,  brake_secondary=True)
    if not (a_both < a_one):
        raise AssertionError("Braking both stars should yield stronger inspiral than braking one.")

def phys_12_timestep_robustness_for_coupled_run():
    """
    Physical (not algorithmic) sanity: coupling result (Δa, Δ|Omega|) should be
    insensitive to modest changes in coupling cadence.
    """
    def evolve(nsteps):
        sim, rebx = new_sim()
        a0 = 0.05
        sim.add(m=1.0, r=RSUN_IN_AU)
        sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)
        sim.move_to_com()
        tctl = rebx.load_force("tides_constant_time_lag"); rebx.add_force(tctl)
        add_magnetic_braking(rebx, K_cgs=1.0e53)
        set_star_spin_and_MB(sim.particles[0], Pspin_days=30.0)
        set_star_spin_and_MB(sim.particles[1], Pspin_days=30.0)
        for p in (sim.particles[0], sim.particles[1]):
            p.params["tctl_k2"] = 0.03
            p.params["tctl_tau"] = 1e-4
        a0_, _ = a_e(sim, 1, 0); w0 = omega_mag(sim.particles[0]) + omega_mag(sim.particles[1])
        integrate_with_spin_sync(sim, tmax=2.0e2, nsteps=nsteps, star_indices=(0,1))
        a1_, _ = a_e(sim, 1, 0); w1 = omega_mag(sim.particles[0]) + omega_mag(sim.particles[1])
        assert_outside_roche(sim, margin=5.0)
        return (a1_ - a0_), (w1 - w0)

    da_fine, dw_fine = evolve(800)
    da_coarse, dw_coarse = evolve(100)
    # Allow a few % difference
    if abs(da_fine - da_coarse) > 0.05*max(1e-10, abs(da_fine)):
        raise AssertionError("Δa should be stable to modest step changes (within ~5%).")
    if abs(dw_fine - dw_coarse) > 0.05*max(1e-10, abs(dw_fine)):
        raise AssertionError("Δ|Omega| should be stable to modest step changes (within ~5%).")

# ----------------------- Runner -----------------------

def main():
    tests = [
        ("01_single_star_MB_removes_L_and_E_preserves_axis", phys_01_single_star_MB_removes_L_and_E_preserves_axis),
        ("02_single_star_no_MB_conserves_L_and_E",            phys_02_single_star_no_MB_conserves_L_and_E),
        ("03_single_star_I_scaling_slows_spindown",           phys_03_single_star_I_scaling_slows_spindown),
        ("04_single_star_saturated_braking_weaker",           phys_04_single_star_saturated_braking_is_weaker_than_unsaturated),
        ("05_binary_MB_only_keeps_orbit_constant",            phys_05_binary_MB_only_keeps_orbit_constant),
        ("06_binary_CTL_only_subsync_inspiral",               phys_06_binary_CTL_only_subsync_inspiral),
        ("07_binary_CTL_only_supersync_expansion",            phys_07_binary_CTL_only_supersync_expansion),
        ("08_binary_CTL_plus_MB_accelerates_inspiral",        phys_08_binary_CTL_plus_MB_accelerates_inspiral),
        ("09_binary_CTL_plus_MB_flip_supersync_to_subsync",   phys_09_binary_CTL_plus_MB_flip_supersync_to_subsync),
        ("10_binary_CTL_only_eccentric_circularization",      phys_10_binary_CTL_only_eccentric_circularization),
        ("11_binary_CTL_plus_MB_both_vs_one_braked",          phys_11_binary_CTL_plus_MB_vs_one_or_both_braked),
        ("12_timestep_robustness_for_coupled_run",            phys_12_timestep_robustness_for_coupled_run),
    ]
    passed = 0
    for name, fn in tests:
        try:
            fn()
            print(f"[PASS] {name}")
            passed += 1
        except AssertionError as e:
            print(f"[FAIL] {name}: {e}")
        except Exception as e:
            print(f"[ERROR] {name}: {e}")
    print(f"\n{passed}/{len(tests)} physical tests passed.")

if __name__ == "__main__":
    main()


/home/mad/anaconda3/envs/rbx2025/lib/python3.13/site-packages/rebound/simulation.py:259: RuntimeWarning: REBOUNDx: Operators that affect particle trajectories with adaptive timesteps can give spurious results. Use sim.ri_ias15.epsilon=0 for fixed timestep with IAS, or use a different integrator.
  warnings.warn(msg[1:], RuntimeWarning)
/home/mad/anaconda3/envs/rbx2025/lib/python3.13/site-packages/rebound/simulation.py:259: RuntimeWarning: At least 10 predictor corrector loops in IAS15 did not converge. This is typically an indication of the timestep being too large.
  warnings.warn(msg[1:], RuntimeWarning)


[PASS] 01_single_star_MB_removes_L_and_E_preserves_axis
[FAIL] 02_single_star_no_MB_conserves_L_and_E: Without MB, spin angular momentum and energy must be constant.
[PASS] 03_single_star_I_scaling_slows_spindown
[PASS] 04_single_star_saturated_braking_weaker
[PASS] 05_binary_MB_only_keeps_orbit_constant
[FAIL] 06_binary_CTL_only_subsync_inspiral: Roche-lobe violation: R1=4.650e-03, RL1=1.863e-02; R2=4.650e-03, RL2=1.863e-02 (margin=5.0)
[FAIL] 07_binary_CTL_only_supersync_expansion: Roche-lobe violation: R1=4.650e-03, RL1=2.127e-02; R2=4.650e-03, RL2=2.127e-02 (margin=5.0)


KeyboardInterrupt: 

In [18]:
sim, rebx = new_sim()
a0 = 0.05
sim.add(m=1.0, r=RSUN_IN_AU)            # star 0
sim.add(m=1.0, a=a0, e=0.0, r=RSUN_IN_AU)  # star 1
sim.move_to_com()

tctl = rebx.load_force("tides_constant_time_lag"); rebx.add_force(tctl)
# Only star 0 dissipates
sim.particles[0].params["tctl_k2"] = 0.03
sim.particles[0].params["tctl_tau"] = 1e-4
sim.particles[1].params["tctl_k2"] = 0.0
sim.particles[1].params["tctl_tau"] = 0.0

# Sub-synchronous: OmegaMag = 0
sim.particles[0].params["OmegaMag"] = 0.

a_start, _ = a_e(sim, 1, 0)
sim.integrate(2.0e3)
a_end, _ = a_e(sim, 1, 0)

assert_outside_roche(sim, margin=5.0)
if not (a_end < a_start):
    raise AssertionError("CTL-only + sub-synchronous spin should shrink the orbit (a decreases).")

TypeError: 'float' object is not subscriptable

In [10]:
"""
One-shot showcase of magnetic braking (with saturation), tidal orbital evolution,
and the spin–orbit coupling in a detached binary star system.

Run:  python binary_magnetic_braking_showcase.py
Requires: rebound, reboundx, numpy

What it demonstrates (in one continuous run):
  • MB torque removes stellar spin L and E_rot (primary only here).
  • Starts in saturated MB (Ω > ω_sat) -> exponential-like spin-down,
    then crosses to unsaturated MB (Ω < ω_sat) -> slower spin-down.
  • CTL tides (on the primary only) cause:
       - initial orbital expansion while Ω > n (super-synchronous),
       - then a sign flip to inspiral once MB makes Ω < n (sub-synchronous).
  • Tidal circularization (e decreases).
  • Angular-momentum budgets: L_spin, J_orb, J_total.
  • Detached throughout: Roche-lobe checks each output (no CE/RLOF physics used).
"""

import math
import numpy as np
import rebound
import reboundx
from dataclasses import dataclass

# ----------------------- Units & constants -----------------------
TWOPI = 2.0*math.pi
G_CODE = TWOPI**2                   # So P=1 yr at 1 AU around 1 Msun
AU_IN_CM = 1.495978707e13
RSUN_IN_CM = 6.957e10
RSUN_IN_AU = RSUN_IN_CM / AU_IN_CM  # ~0.00465047 AU

# ----------------------- Operator/physics knobs (tunable) -----------------------
# Choose parameters to make all effects visible in a short run.
MB_K_CGS       = 1.0e55    # strong MB so spin crosses saturation and n within ~few kyr
MB_OMEGA_SAT   = None      # if None we set it to 2*n0 at runtime (so start saturated)
TCTL_K2        = 0.03
TCTL_TAU_YR    = 1e-4      # large to accelerate tidal effects
K2_GYRATION    = 0.10      # moment-of-inertia factor I = k2 * M * R^2
R_MARGIN_RLOF  = 5.0       # keep R < RL / margin at all times

# Orbit/ICs
A0_AU          = 0.07      # small, but safely detached for R=1 R_sun
E0             = 0.10
M1 = 1.0; R1 = RSUN_IN_AU
M2 = 1.0; R2 = RSUN_IN_AU

# Spin setup (primary only dissipates and is magnetically braked)
SPIN_MULTIPLE_OF_N = 3.0   # start Ω1 = 3 n0 (super-synchronous, saturated)
SECONDARY_MB_ON    = False # keep secondary off to isolate effects (still can set CTL=0)
SECONDARY_CTL_ON   = False # only primary raises/dissipates tides

# Integration cadence
T_END    = 5000.0          # years
N_OUTPUT = 600             # time samples; coupling sync happens at least this often

# ----------------------- Helpers -----------------------

@dataclass
class Snapshot:
    t: float
    a: float
    e: float
    n: float
    RL1: float
    RL2: float
    Ls1: float
    Ls2: float
    Jorb: float
    Jtot: float
    Om1: float
    Om2: float
    regime_sat: int      # 1 if Ω1 > ω_sat, else 0
    super_sync: int      # 1 if Ω1 > n, else 0
    da_dt: float         # finite-difference (per yr) for diagnostics

def new_sim():
    sim = rebound.Simulation()
    sim.G = G_CODE
    sim.integrator = "ias15"
    return sim, reboundx.Extras(sim)

def add_magnetic_braking(rebx, K_cgs=MB_K_CGS, Rsun_code=RSUN_IN_AU, year_code=1.0):
    mb = rebx.load_operator("magnetic_braking")
    rebx.add_operator(mb)
    mb.params["mb_K"]      = float(K_cgs)
    mb.params["mb_Msun"]   = 1.0
    mb.params["mb_Rsun"]   = float(Rsun_code)
    mb.params["mb_year"]   = float(year_code)
    mb.params["mb_Rossby_sat"] = 0.1
    return mb

def set_star_spin_and_MB(p, M=1.0, R=RSUN_IN_AU, Omega_vec=None, Pspin_days=None,
                         k2_gyration=K2_GYRATION, mb_on=True, convective=True,
                         omega_sat=None, tau_conv_days=None):
    p.m = float(M); p.r = float(R)
    I = k2_gyration * p.m * p.r**2
    p.params["I"] = float(I)
    p.params["mb_on"] = 1 if mb_on else 0
    p.params["mb_convective"] = 1 if convective else 0
    if omega_sat is not None:
        p.params["mb_omega_sat"] = float(omega_sat)
    if tau_conv_days is not None:
        p.params["mb_tau_conv"] = float(tau_conv_days)/365.25
    if Omega_vec is not None:
        p.params["Omega"] = np.array(Omega_vec, dtype=float)
    elif Pspin_days is not None:
        Pspin_yr = Pspin_days/365.25
        Om = TWOPI/Pspin_yr
        p.params["Omega"] = np.array([0.0,0.0,Om], dtype=float)
    else:
        p.params["Omega"] = np.array([0.0,0.0,0.0], dtype=float)

def omega_vec_from_params(p):
    v = p.params.get("Omega", None)
    if v is None: return np.array([0.0,0.0,0.0], dtype=float)
    try:
        arr = np.array(v, dtype=float).reshape(-1)
        if arr.size == 3: return arr
    except Exception:
        pass
    if hasattr(v, "x") and hasattr(v, "y") and hasattr(v, "z"):
        return np.array([float(v.x), float(v.y), float(v.z)], dtype=float)
    if isinstance(v, (float,int)):
        return np.array([0.0,0.0,float(v)], dtype=float)
    raise TypeError("Omega parameter not a 3-vector")

def omega_mag(p): return float(np.linalg.norm(omega_vec_from_params(p)))
def L_spin_vec(p): return float(p.params["I"]) * omega_vec_from_params(p)
def L_spin_mag(p): return float(np.linalg.norm(L_spin_vec(p)))

def rel_state(sim, i=0, j=1):
    pi, pj = sim.particles[i], sim.particles[j]
    r = np.array([pj.x-pi.x, pj.y-pi.y, pj.z-pi.z], dtype=float)
    v = np.array([pj.vx-pi.vx, pj.vy-pi.vy, pj.vz-pi.vz], dtype=float)
    return r, v

def J_orbital_vec(sim, i=0, j=1):
    pi, pj = sim.particles[i], sim.particles[j]
    mu = pi.m*pj.m/(pi.m+pj.m)
    r, v = rel_state(sim, i, j)
    return mu*np.cross(r, v)

def J_orbital_mag(sim, i=0, j=1): return float(np.linalg.norm(J_orbital_vec(sim,i,j)))
def J_total_vec(sim): return J_orbital_vec(sim) + L_spin_vec(sim.particles[0]) + L_spin_vec(sim.particles[1])
def J_total_mag(sim): return float(np.linalg.norm(J_total_vec(sim)))

def a_e(sim, i=1, j=0):
    orb = sim.particles[i]#.calculate_orbit(primary=sim.particles[j])
    return float(orb.a), float(orb.e)

def n_mean_motion(sim, a, m_total):  # rad/yr
    return math.sqrt(sim.G * m_total / (a**3))

def roche_lobe_radius(a, q):
    q13 = q**(1.0/3.0)
    return a * 0.49*q13*q13 / (0.6*q13*q13 + math.log(1.0 + q13))

def assert_outside_roche(sim, margin=R_MARGIN_RLOF):
    a, _ = a_e(sim, 1, 0)
    M1, M2 = sim.particles[0].m, sim.particles[1].m
    RL1 = roche_lobe_radius(a, q=M1/M2)
    RL2 = roche_lobe_radius(a, q=M2/M1)
    if not (sim.particles[0].r < RL1/margin and sim.particles[1].r < RL2/margin):
        raise RuntimeError("Roche-lobe violation -> CE/RLOF would occur (we avoid those physics here).")

def sync_OmegaMag_to_Omega(sim, idxs=(0,1)):
    for i in idxs:
        sim.particles[i].params["OmegaMag"] = omega_mag(sim.particles[i])

# ----------------------- Main setup & run -----------------------

def main():
    # Build binary
    sim, rebx = new_sim()
    sim.add(m=M1, r=R1)                 # primary
    sim.add(m=M2, a=A0_AU, e=E0, r=R2)  # secondary
    sim.move_to_com()

    # Add CTL tides (we'll use it only on primary, to get clean sign flips)
    tctl = rebx.load_force("tides_constant_time_lag")
    rebx.add_force(tctl)
    # Primary dissipates
    sim.particles[0].params["tctl_k2"]  = TCTL_K2
    sim.particles[0].params["tctl_tau"] = TCTL_TAU_YR
    # Secondary off (optional)
    if SECONDARY_CTL_ON:
        sim.particles[1].params["tctl_k2"]  = TCTL_K2
        sim.particles[1].params["tctl_tau"] = TCTL_TAU_YR
    else:
        sim.particles[1].params["tctl_k2"]  = 0.0
        sim.particles[1].params["tctl_tau"] = 0.0

    # Add magnetic braking operator
    add_magnetic_braking(rebx, K_cgs=MB_K_CGS)

    # Compute initial mean motion and set spins
    a0, e0 = a_e(sim, 1, 0)
    n0 = n_mean_motion(sim, a0, M1+M2)
    Om1_init = SPIN_MULTIPLE_OF_N * n0
    # If omega_sat not given, set to 2 n0 so we start in saturation (Ω1=3n0>ω_sat=2n0)
    omega_sat = MB_OMEGA_SAT if MB_OMEGA_SAT is not None else 2.0*n0

    # Primary: MB on, CTL on; start super-synchronous and saturated
    set_star_spin_and_MB(sim.particles[0],
                         M=M1, R=R1, Omega_vec=(0,0,Om1_init),
                         k2_gyration=K2_GYRATION, mb_on=True,
                         omega_sat=omega_sat)
    # Secondary: MB optional (default off), CTL optional (set above)
    set_star_spin_and_MB(sim.particles[1],
                         M=M2, R=R2, Omega_vec=(0,0,0.5*n0),   # sub-synchronous, small
                         k2_gyration=K2_GYRATION,
                         mb_on=SECONDARY_MB_ON)

    # Time loop
    times = np.linspace(0.0, T_END, N_OUTPUT)
    snaps = []
    a_prev = a0

    # Record initial budgets
    Jorb0 = J_orbital_mag(sim)
    Jtot0 = J_total_mag(sim)
    Ls0   = L_spin_mag(sim.particles[0]) + L_spin_mag(sim.particles[1])

    # Integrate with MB→CTL coupling
    for t in times:
        sync_OmegaMag_to_Omega(sim, (0,1))
        sim.integrate(t)

        # Diagnostics
        a, e = a_e(sim, 1, 0)
        n  = n_mean_motion(sim, a, M1+M2)
        RL1 = roche_lobe_radius(a, q=M1/M2)
        RL2 = roche_lobe_radius(a, q=M2/M1)
        assert_outside_roche(sim, margin=R_MARGIN_RLOF)

        Om1 = omega_mag(sim.particles[0])
        Om2 = omega_mag(sim.particles[1])
        sat = 1 if Om1 > (sim.particles[0].params.get("mb_omega_sat", 1e300)) else 0
        ss  = 1 if Om1 > n else 0

        Ls1 = L_spin_mag(sim.particles[0])
        Ls2 = L_spin_mag(sim.particles[1])
        Jorb = J_orbital_mag(sim)
        Jtot = J_total_mag(sim)

        da_dt = (a - a_prev)/max(1e-12, (t - (snaps[-1].t if snaps else 0.0)))
        a_prev = a

        snaps.append(Snapshot(t, a, e, n, RL1, RL2, Ls1, Ls2, Jorb, Jtot, Om1, Om2, sat, ss, da_dt))

    # ----------------------- Post-run analysis & summary -----------------------
    # Event times: leave saturation (Ω1 drops below ω_sat), cross corotation (Ω1 < n),
    # sign flip of da/dt from + to -
    t_leave_sat = next((s.t for s in snaps if s.regime_sat == 0), None)
    t_cross_n   = next((s.t for s in snaps if s.super_sync == 0), None)
    t_da_flip   = None
    for k in range(1, len(snaps)):
        if snaps[k-1].da_dt > 0.0 and snaps[k].da_dt < 0.0:
            t_da_flip = snaps[k].t
            break

    a_init, e_init = snaps[0].a, snaps[0].e
    a_min = min(s.a for s in snaps)
    a_max = max(s.a for s in snaps)
    e_final = snaps[-1].e
    Om1_init, Om1_final = snaps[0].Om1, snaps[-1].Om1

    Jorb1 = snaps[-1].Jorb; Jtot1 = snaps[-1].Jtot
    Ls1_end = snaps[-1].Ls1 + snaps[-1].Ls2

    print("\n=== Magnetic Braking + CTL Showcase (Detached Binary) ===")
    print(f"Masses: M1={M1:.3f} Msun, M2={M2:.3f} Msun | Radii: R1=R2={RSUN_IN_AU:.5f} AU")
    print(f"Initial orbit: a0={a0:.5f} AU, e0={e0:.3f}, P0={2*math.pi/n0:.4f} yr (~{365.25*2*math.pi/n0:.2f} d)")
    print(f"Initial spins: Ω1={Om1_init:.2f} rad/yr ({Om1_init/n0:.2f} n0), Ω2={snaps[0].Om2:.2f} rad/yr")
    print(f"MB: K={MB_K_CGS:.2e} cgs, ω_sat={'{:.2f}'.format(omega_sat)} rad/yr (≈ {omega_sat/n0:.2f} n0)")
    print(f"CTL (primary only): k2={TCTL_K2}, τ={TCTL_TAU_YR} yr")
    print(f"Detached check: R1/RL1_init={R1/snaps[0].RL1:.3f}, R2/RL2_init={R2/snaps[0].RL2:.3f} (must be ≪1)")
    print("-----------------------------------------------------------")
    print("Key events:")
    print(f"  Leave saturation (Ω1 < ω_sat):       t ≈ {t_leave_sat:.1f} yr" if t_leave_sat is not None else "  [did not leave saturation]")
    print(f"  Cross corotation (Ω1 < n):           t ≈ {t_cross_n:.1f} yr"   if t_cross_n   is not None else "  [did not cross Ω=n]")
    print(f"  Orbit flip (da/dt: + → -):           t ≈ {t_da_flip:.1f} yr"   if t_da_flip   is not None else "  [no flip detected]")
    print("-----------------------------------------------------------")
    print(f"Orbital evolution: a_min={a_min:.6f} AU, a_max={a_max:.6f} AU | a_final={snaps[-1].a:.6f} AU")
    print(f"                  e: e_init={e_init:.4f} → e_final={e_final:.4f}  (circularization expected)")
    print(f"Spin evolution:    Ω1: {Om1_init:.2f} → {Om1_final:.2f} rad/yr  (MB removes spin L, crosses regimes)")
    print("-----------------------------------------------------------")
    print("Angular-momentum budget (magnitudes):")
    print(f"  J_orb:   {Jorb0:.6e} → {Jorb1:.6e}  (sign set by Ω−n via CTL)")
    print(f"  L_spinΣ: {Ls0:.6e} → {Ls1_end:.6e}  (MB drains spin)")
    print(f"  J_total: {Jtot0:.6e} → {Jtot1:.6e}  (MB extracts AM from system)")
    print("-----------------------------------------------------------")
    print("CSV written: magnetic_braking_binary_showcase.csv (time series)")

    # Write CSV for quick plotting/inspection
    hdr = ("t_yr,a_AU,e,n_radyr,RL1_AU,RL2_AU,Ls1,Ls2,Jorb,Jtot,"
           "Omega1,Omega2,MB_saturated(super=1),super_synchronous(1),da_dt_AU_per_yr")
    arr = np.array([[s.t, s.a, s.e, s.n, s.RL1, s.RL2, s.Ls1, s.Ls2, s.Jorb, s.Jtot,
                     s.Om1, s.Om2, s.regime_sat, s.super_sync, s.da_dt] for s in snaps], dtype=float)
    np.savetxt("magnetic_braking_binary_showcase.csv", arr, delimiter=",", header=hdr)

if __name__ == "__main__":
    main()



=== Magnetic Braking + CTL Showcase (Detached Binary) ===
Masses: M1=1.000 Msun, M2=1.000 Msun | Radii: R1=R2=0.00465 AU
Initial orbit: a0=0.07000 AU, e0=0.100, P0=0.0131 yr (~4.78 d)
Initial spins: Ω1=1439.36 rad/yr (3.00 n0), Ω2=239.89 rad/yr
MB: K=1.00e+55 cgs, ω_sat=959.57 rad/yr (≈ 2.00 n0)
CTL (primary only): k2=0.03, τ=0.0001 yr
Detached check: R1/RL1_init=0.175, R2/RL2_init=0.175 (must be ≪1)
-----------------------------------------------------------
Key events:
  Leave saturation (Ω1 < ω_sat):       t ≈ 8.3 yr
  Cross corotation (Ω1 < n):           t ≈ 8.3 yr
  Orbit flip (da/dt: + → -):           t ≈ 16.7 yr
-----------------------------------------------------------
Orbital evolution: a_min=0.067608 AU, a_max=0.070007 AU | a_final=0.067608 AU
                  e: e_init=0.1000 → e_final=0.0866  (circularization expected)
Spin evolution:    Ω1: 1439.36 → 17.43 rad/yr  (MB removes spin L, crosses regimes)
-----------------------------------------------------------
Angular-mo

In [13]:
"""
Plotting for the Magnetic Braking + CTL showcase (detached binary).

Input: magnetic_braking_binary_showcase.csv (from the simulation script)
Output: figures/*.pdf and figures/*.png (publication-ready)

Each figure is a single, standalone plot (no subplots) with default Matplotlib
styling and colors. Event markers show:
  • Leaving saturation (Ω1 < ω_sat)
  • Crossing corotation (Ω1 < n)
  • Sign flip in da/dt (expansion → inspiral)

Run:
  python plot_magnetic_braking_showcase.py [optional_path_to_csv]
"""

import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# -------------------- Config --------------------
DEFAULT_CSV = "magnetic_braking_binary_showcase.csv"
OUTDIR = "figures"
DPI = 300          # PNG dpi
FIGSIZE = (6.0, 4.0)  # inches, per-figure

# -------------------- I/O --------------------
csv_path =  "magnetic_braking_binary_showcase.csv" #sys.argv[1] if len(sys.argv) > 1 else DEFAULT_CSV
if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"Could not find '{csv_path}'. Run the simulation script first "
        "to generate magnetic_braking_binary_showcase.csv, or pass a path."
    )

# Column order written by the simulation:
# t_yr, a_AU, e, n_radyr, RL1_AU, RL2_AU, Ls1, Ls2, Jorb, Jtot,
# Omega1, Omega2, MB_saturated(1/0), super_synchronous(1/0), da_dt_AU_per_yr
data = np.loadtxt(csv_path, delimiter=",", skiprows=1)
t       = data[:, 0]
a       = data[:, 1]
e       = data[:, 2]
n       = data[:, 3]
RL1     = data[:, 4]
RL2     = data[:, 5]
Ls1     = data[:, 6]
Ls2     = data[:, 7]
Jorb    = data[:, 8]
Jtot    = data[:, 9]
Omega1  = data[:,10]
Omega2  = data[:,11]
sat     = data[:,12].astype(int)  # 1 iff Ω1 > ω_sat
ss      = data[:,13].astype(int)  # 1 iff Ω1 > n
da_dt   = data[:,14]

# If you used 1 R_sun in the sim, this is the R in AU from the sim header; adjust if needed.
RSUN_IN_AU = 6.957e10 / 1.495978707e13
R1 = RSUN_IN_AU
R2 = RSUN_IN_AU

Omega_over_n = Omega1 / n
Lspin_sum    = Ls1 + Ls2
R1_over_RL1  = R1 / RL1
R2_over_RL2  = R2 / RL2

# Event times
def first_transition_time(flag_array, from_val, to_val):
    idx = np.where((flag_array[:-1] == from_val) & (flag_array[1:] == to_val))[0]
    return None if idx.size == 0 else t[idx[0]+1]

t_leave_saturation = first_transition_time(sat, 1, 0)   # Ω1 crosses below ω_sat
t_cross_corotation = first_transition_time(ss,  1, 0)   # Ω1 crosses below n
# da/dt sign flip: + to -
idx_flip = np.where((da_dt[:-1] > 0.0) & (da_dt[1:] < 0.0))[0]
t_da_flip = None if idx_flip.size == 0 else t[idx_flip[0]+1]

# -------------------- Utilities --------------------
os.makedirs(OUTDIR, exist_ok=True)

def savefig(name):
    pdf = os.path.join(OUTDIR, f"{name}.pdf")
    png = os.path.join(OUTDIR, f"{name}.png")
    plt.savefig(pdf, bbox_inches="tight")
    plt.savefig(png, dpi=DPI, bbox_inches="tight")
    print(f"Saved: {pdf}\n       {png}")
    plt.close()

def add_event_markers(ax, include_da_flip=True):
    if t_leave_saturation is not None:
        ax.axvline(t_leave_saturation, linestyle="--", linewidth=1.0, label="leave saturation")
    if t_cross_corotation is not None:
        ax.axvline(t_cross_corotation, linestyle=":", linewidth=1.0, label="Ω₁ = n")
    if include_da_flip and t_da_flip is not None:
        ax.axvline(t_da_flip, linestyle="-.", linewidth=1.0, label="da/dt flip")

# -------------------- Fig 1: a(t) --------------------
plt.figure(figsize=FIGSIZE)
plt.plot(t, a, linewidth=1.5, label="a(t)")
add_event_markers(plt.gca(), include_da_flip=True)
plt.xlabel("Time (yr)")
plt.ylabel("Semimajor axis, a (AU)")
plt.title("Orbital evolution: semimajor axis")
plt.legend()
plt.tight_layout()
savefig("fig1_a_of_t")

# -------------------- Fig 2: e(t) --------------------
plt.figure(figsize=FIGSIZE)
plt.plot(t, e, linewidth=1.5, label="e(t)")
add_event_markers(plt.gca(), include_da_flip=False)
plt.xlabel("Time (yr)")
plt.ylabel("Eccentricity, e")
plt.title("Orbital circularization")
plt.legend()
plt.tight_layout()
savefig("fig2_e_of_t")

# -------------------- Fig 3: Ω1(t) and n(t) --------------------
plt.figure(figsize=FIGSIZE)
plt.plot(t, Omega1, linewidth=1.5, label=r"$\Omega_1(t)$")
plt.plot(t, n,      linewidth=1.2, label=r"$n(t)$")
add_event_markers(plt.gca(), include_da_flip=False)
plt.xlabel("Time (yr)")
plt.ylabel(r"Angular frequency (rad yr$^{-1}$)")
plt.title("Spin–orbit rates")
plt.legend()
plt.tight_layout()
savefig("fig3_spin_vs_meanmotion")

# -------------------- Fig 4: Ω1/n --------------------
plt.figure(figsize=FIGSIZE)
plt.plot(t, Omega_over_n, linewidth=1.5, label=r"$\Omega_1/n$")
plt.axhline(1.0, linestyle="--", linewidth=1.0, label="co-rotation")
add_event_markers(plt.gca(), include_da_flip=False)
plt.xlabel("Time (yr)")
plt.ylabel(r"$\Omega_1/n$")
plt.title("Co-rotation crossing")
plt.legend()
plt.tight_layout()
savefig("fig4_Omega_over_n")

# -------------------- Fig 5: Angular-momentum budgets --------------------
plt.figure(figsize=FIGSIZE)
plt.plot(t, Jorb,      linewidth=1.5, label=r"$J_{\rm orb}$")
plt.plot(t, Lspin_sum, linewidth=1.2, label=r"$L_{\rm spin,1}+L_{\rm spin,2}$")
plt.plot(t, Jtot,      linewidth=1.2, label=r"$J_{\rm total}$")
add_event_markers(plt.gca(), include_da_flip=False)
plt.xlabel("Time (yr)")
plt.ylabel("Angular momentum (code units)")
plt.title("Angular-momentum budgets")
plt.legend()
plt.tight_layout()
savefig("fig5_angular_momentum_budgets")

# -------------------- Fig 6: da/dt (expansion → inspiral) --------------------
plt.figure(figsize=FIGSIZE)
plt.plot(t, da_dt, linewidth=1.5, label=r"$\mathrm{d}a/\mathrm{d}t$")
plt.axhline(0.0, linestyle="--", linewidth=1.0, label="zero")
add_event_markers(plt.gca(), include_da_flip=True)
plt.xlabel("Time (yr)")
plt.ylabel(r"$\mathrm{d}a/\mathrm{d}t$ (AU yr$^{-1}$)")
plt.title("Sign of orbital evolution")
plt.legend()
plt.tight_layout()
savefig("fig6_da_dt")

# -------------------- Fig 7: Detached safety (R/RL) --------------------
plt.figure(figsize=FIGSIZE)
plt.plot(t, R1_over_RL1, linewidth=1.5, label=r"$R_1/R_{\rm L,1}$")
plt.plot(t, R2_over_RL2, linewidth=1.2, label=r"$R_2/R_{\rm L,2}$")
plt.axhline(1.0, linestyle="--", linewidth=1.0, label="Roche-lobe limit")
plt.xlabel("Time (yr)")
plt.ylabel("Radius / Roche-lobe radius")
plt.title("Detached condition check")
plt.legend()
plt.tight_layout()
savefig("fig7_detached_condition")

# -------------------- Fig 8 (optional): secondary spin --------------------
plt.figure(figsize=FIGSIZE)
plt.plot(t, Omega2, linewidth=1.5, label=r"$\Omega_2(t)$")
plt.xlabel("Time (yr)")
plt.ylabel(r"Angular frequency (rad yr$^{-1}$)")
plt.title("Secondary spin (for completeness)")
plt.legend()
plt.tight_layout()
savefig("fig8_secondary_spin")

# -------------------- Summary text file (events) --------------------
events_txt = os.path.join(OUTDIR, "events_summary.txt")
with open(events_txt, "w") as f:
    f.write("Magnetic Braking + CTL Showcase — Key Events (times in yr)\n")
    f.write("----------------------------------------------------------\n")
    f.write(f"Leave saturation (Ω1 < ω_sat):  {t_leave_saturation if t_leave_saturation is not None else '—'}\n")
    f.write(f"Cross corotation (Ω1 < n):      {t_cross_corotation if t_cross_corotation is not None else '—'}\n")
    f.write(f"da/dt sign flip (+ → −):        {t_da_flip if t_da_flip is not None else '—'}\n")
print(f"Saved: {events_txt}")

print("\nDone. Figures are in the 'figures/' directory.")


Saved: figures/fig1_a_of_t.pdf
       figures/fig1_a_of_t.png
Saved: figures/fig2_e_of_t.pdf
       figures/fig2_e_of_t.png
Saved: figures/fig3_spin_vs_meanmotion.pdf
       figures/fig3_spin_vs_meanmotion.png
Saved: figures/fig4_Omega_over_n.pdf
       figures/fig4_Omega_over_n.png
Saved: figures/fig5_angular_momentum_budgets.pdf
       figures/fig5_angular_momentum_budgets.png
Saved: figures/fig6_da_dt.pdf
       figures/fig6_da_dt.png
Saved: figures/fig7_detached_condition.pdf
       figures/fig7_detached_condition.png
Saved: figures/fig8_secondary_spin.pdf
       figures/fig8_secondary_spin.png
Saved: figures/events_summary.txt

Done. Figures are in the 'figures/' directory.
